In [1]:
from pyspark.sql.functions import col, sum as spark_sum


# Read Bronze Delta table

df_bronze = spark.read.format("delta")\
                 .table("bronze_patient_records")

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 3, Finished, Available, Finished, False)

In [2]:
print(f"Bronze row count: {df_bronze.count()}")
df_bronze.printSchema()


StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 4, Finished, Available, Finished, False)

Bronze row count: 55500
root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Blood_Type: string (nullable = true)
 |-- Medical_Condition: string (nullable = true)
 |-- Date_of_Admission: date (nullable = true)
 |-- Doctor: string (nullable = true)
 |-- Hospital: string (nullable = true)
 |-- Insurance_Provider: string (nullable = true)
 |-- Billing_Amount: double (nullable = true)
 |-- Room_Number: integer (nullable = true)
 |-- Admission_Type: string (nullable = true)
 |-- Discharge_Date: date (nullable = true)
 |-- Medication: string (nullable = true)
 |-- Test_Results: string (nullable = true)



In [3]:
# Check null counts in each column

null_counts = df_bronze.select([
    spark_sum( col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])

null_counts.show(truncate = False)

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 5, Finished, Available, Finished, False)

+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|Name|Age|Gender|Blood_Type|Medical_Condition|Date_of_Admission|Doctor|Hospital|Insurance_Provider|Billing_Amount|Room_Number|Admission_Type|Discharge_Date|Medication|Test_Results|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|0   |0  |0     |0         |0                |0                |0     |0       |0                 |0             |0          |0             |0             |0         |0           |
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+



In [4]:
from pyspark.sql.functions import datediff, round, upper, trim

# Basic Transformation and finding length_of_Stay

df_silver = df_bronze\
    .withColumn("length_of_Stay", 
                datediff(col("Discharge_Date"), col("Date_of_Admission")))\
    .withColumn("Billing_Amount",
                round(col("Billing_Amount"), 2))\
    .withColumn("Gender",
                upper(trim(col("Gender"))))\
    .withColumn("Medical_Condition",
                upper(trim(col("Medical_Condition"))))\
    .withColumn("Admission_Type",
                upper(trim(col("Admission_Type"))))\
    .withColumn("Test_Results",
                upper(trim(col("test_Results"))))


print(f"Silver row count: {df_silver.count()}")
df_silver.show(5, truncate=False)

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 6, Finished, Available, Finished, False)

Silver row count: 55500
+-------------+---+------+----------+-----------------+-----------------+----------------+--------------------------+------------------+--------------+-----------+--------------+--------------+-----------+------------+--------------+
|Name         |Age|Gender|Blood_Type|Medical_Condition|Date_of_Admission|Doctor          |Hospital                  |Insurance_Provider|Billing_Amount|Room_Number|Admission_Type|Discharge_Date|Medication |Test_Results|length_of_Stay|
+-------------+---+------+----------+-----------------+-----------------+----------------+--------------------------+------------------+--------------+-----------+--------------+--------------+-----------+------------+--------------+
|Bobby JacksOn|30 |MALE  |B-        |CANCER           |2024-01-31       |Matthew Smith   |Sons and Miller           |Blue Cross        |18856.28      |328        |URGENT        |2024-02-02    |Paracetamol|NORMAL      |2             |
|LesLie TErRy |62 |MALE  |A+        |OBE

In [5]:
from pyspark.sql.functions import initcap

# Fix Name and Doctor column casing

df_silver = df_silver \
    .withColumn("Name", initcap(col("Name")))\
    .withColumn("Doctor", initcap(col("Doctor")))

display(df_silver)

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4dca4578-795f-44ca-9e47-60b98b7a973d)

In [6]:
from pyspark.sql.functions import when, months_between, current_date

# 1. Creating Age Groups- Minor/Young Adult/Middle Aged/Senior

df_silver = df_silver.withColumn("Age_Group",
    when(col("Age") < 18, "Minor")
    .when( (col("Age") >=18) & (col("Age") <35), "Young Adult")
    .when( (col("Age") >=35) & (col("Age") <60), "Middle Aged")
    .otherwise("Senior")
)



StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 8, Finished, Available, Finished, False)

In [7]:
# 2. Billing Category - low/medium/high


df_silver = df_silver.withColumn("Billing_Category",
    when(col("Billing_Amount") < 10000, "Low")
    .when( (col("Billing_Amount") >=10000) & (col("Billing_Amount") <30000), "Medium")
    .otherwise("High")
)



StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 9, Finished, Available, Finished, False)

In [8]:
# 3. Stay Category - short/medium/long

df_silver = df_silver.withColumn("Stay_Category",
    when(col("Length_of_Stay") <= 7, "Short Stay")
    .when( (col("Length_of_Stay") >=7) & (col("Length_of_Stay") <- 20), "Medium Stay")
    .otherwise("Long Stay")
)

df_silver = df_silver.withColumn("Stay_Category",
    when(col("Length_of_Stay") <= 7, "Short Stay")
    .when((col("Length_of_Stay") > 7) & (col("Length_of_Stay") <= 20), "Medium Stay")
    .otherwise("Long Stay"))

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 10, Finished, Available, Finished, False)

In [11]:
display(df_silver.select("Name", "Age", "Age_Group", "Billing_Amount", "Billing_Category", "Length_of_Stay", "Stay_Category").limit(10))

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ae38e53-e47c-497c-b964-f2e8865cd933)

In [13]:
from pyspark.sql.functions import lit, expr

# 4. Risk Score - Medical condition + Test Results combination

df_silver = df_silver.withColumn("Risk_Flag",
    when( (col("Test_Results") == "ABNORMAL") & (col("Admission_type") == "URGENT"),"High Risk" )
    .when( (col("Test_Results") == "ABNORMAL") | (col("Admission_type") == "URGENT"),"Medium Risk" )
    .otherwise("Low Risk")
)

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 15, Finished, Available, Finished, False)

In [16]:
# 5. Insurance Coverage Flag - Is billing amount reasonable?

df_silver =  df_silver.withColumn("High_Billing_Flag",
 when( col("Billing_Amount") > 40000, lit(True))
 .otherwise(lit(False))
)

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 18, Finished, Available, Finished, False)

In [17]:
display(df_silver.select(
    "Name", "Medical_Condition", "Test_Results", "Admission_Type", "Risk_Flag", "Billing_Amount", "High_Billing_Flag"
).limit(10)
)

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 512ec3ca-dfe9-470a-afd4-aa0c5ffecfa0)

In [18]:
# Save as Silver Delta table

df_silver.write.\
    format("delta")\
    .mode("overwrite")\
    .saveAsTable("silver_patient_records")

print("Silver table saved Successfully!")


StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 20, Finished, Available, Finished, False)

Silver table saved Successfully!


In [19]:
print(f"Total rows: {df_silver.count()}")
print(f"Total column: {len(df_silver.columns)}")

StatementMeta(, 840c9f3e-81fe-4333-88a9-57d2cb0cb73d, 21, Finished, Available, Finished, False)

Total rows: 55500
Total column: 22
